In [ ]:
import json
import re
import sys
import html
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "collection_notebooks" else Path.cwd()
project_root_str = str(PROJECT_ROOT.resolve())

if project_root_str not in sys.path:
    sys.path.append(project_root_str)

from src.utils.hackernews_client import hackerNewsItem, searchHackerNewsStories

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "collection_notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data/raw/hackernews"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/hackernews"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MAX_STORIES_PER_QUERY = 30
MAX_COMMENTS_PER_STORY = 30
MAX_COMMENT_DEPTH = 1

SEARCH_QUERIES = [
    "GitHub Copilot",
    "Claude Code",
    "OpenAI Codex",
    "AI coding assistant",
    "vibe coding",
    "AI replacing programmers",
    "AI code security privacy",
    "AI software developer productivity",
]

In [ ]:
def unix_to_iso(unix_time):
    """
    Convert Hacker News Unix timestamp into readable ISO date.
    """
    if not unix_time:
        return None

    return datetime.fromtimestamp(unix_time, tz=timezone.utc).strftime("%Y-%m-%d")


def fetch_comment_tree(commentIds, maxComments=200, maxDepth=3, parentId=None, depth=1):
    """
    Fetch comments from Hacker News.

    This also collects replies to comments up to maxDepth.
    """

    comments = []

    if not commentIds or maxComments <= 0 or depth > maxDepth:
        return comments

    for commentId in commentIds:
        if len(comments) >= maxComments:
            break

        try:
            comment = hackerNewsItem(commentId)
        except Exception as e:
            print(f"Could not fetch comment {commentId}: {e}")
            continue

        if not comment:
            continue

        if comment.get("deleted") or comment.get("dead"):
            continue

        if comment.get("type") == "comment":
            comment_record = {
                "commentId": comment.get("id"),
                "parentId": parentId,
                "author": comment.get("by"),
                "text": comment.get("text") or "",
                "createdAt": unix_to_iso(comment.get("time")),
                "depth": depth,
                "childCommentCount": len(comment.get("kids", [])),
            }

            comments.append(comment_record)

            remaining = maxComments - len(comments)

            child_comments = fetch_comment_tree(
                comment.get("kids", []),
                maxComments=remaining,
                maxDepth=maxDepth,
                parentId=comment.get("id"),
                depth=depth + 1,
            )

            comments.extend(child_comments)

    return comments[:maxComments]


def fetchHackerNewsData(
    searchQuery,
    maxStories=50,
    maxCommentsPerStory=200,
    maxCommentDepth=3,
    outputFile="hackernewsDataDump.json"
):
    """
    Search Hacker News stories for a query, then collect each story and comments.
    """

    story_hits = searchHackerNewsStories(searchQuery, maxStories=maxStories)
    story_threads = []
    seen_story_ids = set()

    for searchRank, hit in enumerate(story_hits, start=1):
        story_id = int(hit.get("objectID"))

        if story_id in seen_story_ids:
            continue

        seen_story_ids.add(story_id)

        try:
            story = hackerNewsItem(story_id)
        except Exception as e:
            print(f"Could not fetch story {story_id}: {e}")
            continue

        if not story:
            continue

        if story.get("type") != "story":
            continue

        if story.get("deleted") or story.get("dead"):
            continue

        comments = fetch_comment_tree(
            story.get("kids", []),
            maxComments=maxCommentsPerStory,
            maxDepth=maxCommentDepth,
            parentId=story_id,
            depth=1,
        )

        story_threads.append({
            "sourceQuery": searchQuery,
            "searchRank": searchRank,
            "story": story,
            "comments": comments,
        })

    data = {
        "source": "hackernews",
        "searchQuery": searchQuery,
        "storyThreads": story_threads,
    }

    with open(outputFile, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(story_threads)} Hacker News stories to {outputFile}")

    return data

In [ ]:
print("Hacker News collection config:")
print("Search queries:", SEARCH_QUERIES)
print("MAX_STORIES_PER_QUERY:", MAX_STORIES_PER_QUERY)
print("MAX_COMMENTS_PER_STORY:", MAX_COMMENTS_PER_STORY)
print("MAX_COMMENT_DEPTH:", MAX_COMMENT_DEPTH)

for index, searchQuery in enumerate(SEARCH_QUERIES, start=1):
    output_file = RAW_DIR / f"hackernews_ai_coding_{index}.json"

    print(f"\nCollecting '{searchQuery}' -> {output_file}")

    fetchHackerNewsData(
        searchQuery,
        maxStories=MAX_STORIES_PER_QUERY,
        maxCommentsPerStory=MAX_COMMENTS_PER_STORY,
        maxCommentDepth=MAX_COMMENT_DEPTH,
        outputFile=str(output_file),
    )

In [ ]:
raw_files = sorted(RAW_DIR.glob("hackernews_ai_coding_*.json"))

if not raw_files:
    raise FileNotFoundError("No raw Hacker News files found. Run the collection cell first.")

story_rows = []
comment_rows = []
text_rows = []

seen_story_ids = set()
seen_comment_ids = set()

for json_file in raw_files:
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for thread in data.get("storyThreads", []):
        story = thread.get("story", {})
        story_id = story.get("id")
        story_key = f"hn_{story_id}"

        if story_id not in seen_story_ids:
            seen_story_ids.add(story_id)

            story_rows.append({
                "sourceQuery": thread.get("sourceQuery"),
                "searchRank": thread.get("searchRank"),
                "storyKey": story_key,
                "storyId": story_id,
                "title": story.get("title", ""),
                "text": story.get("text") or "",
                "author": story.get("by"),
                "score": story.get("score", 0),
                "createdAt": unix_to_iso(story.get("time")),
                "url": story.get("url"),
                "hnUrl": f"https://news.ycombinator.com/item?id={story_id}",
                "commentCount": story.get("descendants", 0),
                "collectedCommentCount": len(thread.get("comments", [])),
            })

            text_rows.append({
                "source": "hackernews",
                "sourceQuery": thread.get("sourceQuery"),
                "threadKey": story_key,
                "recordType": "story",
                "recordId": story_id,
                "author": story.get("by"),
                "createdAt": unix_to_iso(story.get("time")),
                "text": f"{story.get('title', '')}\n\n{story.get('text') or ''}",
                "url": f"https://news.ycombinator.com/item?id={story_id}",
            })

        for comment in thread.get("comments", []):
            comment_id = comment.get("commentId")

            if comment_id in seen_comment_ids:
                continue

            seen_comment_ids.add(comment_id)

            comment_rows.append({
                "sourceQuery": thread.get("sourceQuery"),
                "storyKey": story_key,
                "storyId": story_id,
                "storyTitle": story.get("title", ""),
                "commentId": comment_id,
                "parentId": comment.get("parentId"),
                "commentAuthor": comment.get("author"),
                "commentText": comment.get("text") or "",
                "commentCreatedAt": comment.get("createdAt"),
                "commentDepth": comment.get("depth"),
                "childCommentCount": comment.get("childCommentCount", 0),
                "hnUrl": f"https://news.ycombinator.com/item?id={comment_id}",
            })

            text_rows.append({
                "source": "hackernews",
                "sourceQuery": thread.get("sourceQuery"),
                "threadKey": story_key,
                "recordType": "comment",
                "recordId": comment_id,
                "author": comment.get("author"),
                "createdAt": comment.get("createdAt"),
                "text": comment.get("text") or "",
                "url": f"https://news.ycombinator.com/item?id={comment_id}",
            })

stories_df = pd.DataFrame(story_rows)
comments_df = pd.DataFrame(comment_rows)
text_df = pd.DataFrame(text_rows)

# Save only the useful flattened files
stories_df.to_csv(PROCESSED_DIR / "hackernews_stories.csv", index=False)
comments_df.to_csv(PROCESSED_DIR / "hackernews_comments.csv", index=False)

print("Stories:", stories_df.shape)
print("Comments:", comments_df.shape)
print("Text records:", text_df.shape)

stories_df.head()

In [ ]:
print("Hacker News dataset summary")
print("---------------------------")

print("Number of stories:", len(stories_df))
print("Number of comments:", len(comments_df))
print("Number of text records:", len(text_df))

if len(stories_df):
    stories_df["createdAt"] = pd.to_datetime(stories_df["createdAt"], errors="coerce")
    print("Story date range:", stories_df["createdAt"].min().date(), "to", stories_df["createdAt"].max().date())

if len(comments_df):
    comments_df["commentCreatedAt"] = pd.to_datetime(comments_df["commentCreatedAt"], errors="coerce")
    print("Comment date range:", comments_df["commentCreatedAt"].min().date(), "to", comments_df["commentCreatedAt"].max().date())

In [ ]:
def clean_hackernews_text(text):
    """
    Clean Hacker News story/comment text.
    """

    text = str(text)

    # Decode HTML entities such as &#x2F;, &amp;, &quot;
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Keep letters, numbers, spaces and apostrophes
    text = re.sub(r"[^A-Za-z0-9\s']", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


text_df = text_df[text_df["text"].notna()].copy()

text_df["textClean"] = text_df["text"].apply(clean_hackernews_text)
text_df["textLower"] = text_df["textClean"].str.lower()
text_df["textLength"] = text_df["textClean"].str.len()

text_df = text_df[text_df["textLength"] > 0].copy()

text_df = text_df.drop_duplicates(
    subset=["threadKey", "recordType", "recordId", "textClean"]
).copy()

text_df.to_csv(PROCESSED_DIR / "hackernews_text_cleaned.csv", index=False)

print("Clean text records:", text_df.shape)

text_df.head()

In [ ]:
relevance_keywords = [
    "copilot",
    "cursor",
    "claude",
    "codex",
    "ai",
    "agent",
    "coding",
    "programmer",
    "developer",
    "productivity",
    "trust",
    "hallucinat",
    "bug",
    "security",
    "privacy",
    "cost",
    "subscription",
    "ownership",
]

for keyword in relevance_keywords:
    count = text_df["textLower"].str.contains(re.escape(keyword), regex=True).sum()
    print(f"{keyword}: {count}")

In [ ]:
stop_words = {
    "the", "and", "that", "for", "this", "you", "with", "but", "not",
    "are", "have", "can", "they", "was", "more", "will", "how", "all",
    "your", "there", "from", "what", "just", "would", "like", "use",
    "about", "into", "then", "than", "when", "which", "their", "them",
    "these", "those", "also", "been", "because", "could", "should",
    "were", "has", "had", "his", "her", "our", "out", "get", "got",
    "one", "two", "some", "any", "who", "why", "where", "way", "see",
    "make", "much", "many", "very", "really", "even", "still", "only",
    "does", "did", "doing", "done", "over", "under", "between", "after",
    "before", "while", "through", "using", "used", "being", "same",
    "well", "think", "people", "thing", "things"
}

word_counter = Counter()

for text in text_df["textLower"]:
    words = re.findall(r"[a-z][a-z0-9_]{2,}", str(text))

    useful_words = []

    for word in words:
        if word not in stop_words:
            useful_words.append(word)

    word_counter.update(useful_words)

common_words = word_counter.most_common(30)

common_terms_df = pd.DataFrame(common_words, columns=["term", "frequency"])

common_terms_df

In [ ]:
user_story_rows = []

for _, row in comments_df.iterrows():
    comment_author = row.get("commentAuthor")
    story_key = row.get("storyKey")

    if pd.notna(comment_author) and pd.notna(story_key):
        user_story_rows.append({
            "source": f"user_{comment_author}",
            "target": story_key,
            "relationship": "commented_on_story",
            "weight": 1,
        })

user_story_edges = pd.DataFrame(user_story_rows)

if len(user_story_edges):
    user_story_edges = (
        user_story_edges
        .groupby(["source", "target", "relationship"], as_index=False)
        .agg(weight=("weight", "sum"))
    )

user_story_edges.to_csv(PROCESSED_DIR / "hackernews_user_story_edges.csv", index=False)

print("User-story edge table:", user_story_edges.shape)

user_story_edges.head()

In [ ]:
# Final clean export
# Removes old/intermediate CSVs and keeps only the final useful files.

for csv_file in PROCESSED_DIR.glob("*.csv"):
    csv_file.unlink()

stories_df.to_csv(PROCESSED_DIR / "hackernews_stories.csv", index=False)
comments_df.to_csv(PROCESSED_DIR / "hackernews_comments.csv", index=False)
text_df.to_csv(PROCESSED_DIR / "hackernews_text_cleaned.csv", index=False)
user_story_edges.to_csv(PROCESSED_DIR / "hackernews_user_story_edges.csv", index=False)

print("Clean Hacker News processed files saved:")

for csv_file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(csv_file.name)